# PubMed -> Vertex AI Function-Calling SFT Prep

Converts local tool-calling SFT JSONL into Gemini supervised tuning JSONL format for Model Garden / Agent Platform.

Source expected: `pubmed_oncologist_v2_tool_sft_messages.jsonl`

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/home/spark/projects/training/pubmed')
SRC_FILE = PROJECT_ROOT / 'data' / 'training-data' / 'pubmed_oncologist_v2_tool_sft_messages.jsonl'
OUT_DIR = PROJECT_ROOT / 'data' / 'training-data' / 'vertex_model_garden'
OUT_FILE = OUT_DIR / 'pubmed_vertex_sft_function_calling.jsonl'

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Source:', SRC_FILE)
print('Output:', OUT_FILE)

In [ ]:
import json
from typing import Any, Dict, List

def _try_json(s: str) -> Any:
    if not isinstance(s, str):
        return s
    s = s.strip()
    if not s:
        return ''
    try:
        return json.loads(s)
    except Exception:
        return s

def _to_tools(tools: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    if not tools:
        return []

    declarations = []
    for t in tools:
        fn = t.get('function', {})
        if not fn:
            continue
        decl = {
            'name': fn.get('name'),
            'description': fn.get('description', ''),
            'parameters': fn.get('parameters', {'type': 'object', 'properties': {}}),
        }
        declarations.append(decl)

    return [{'functionDeclarations': declarations}] if declarations else []

def _msg_to_content(msg: Dict[str, Any]) -> Dict[str, Any]:
    role = msg.get('role')

    if role == 'user':
        return {'role': 'user', 'parts': [{'text': msg.get('content', '')}]}

    if role == 'assistant':
        parts = []
        for tc in msg.get('tool_calls', []):
            fn = tc.get('function', {})
            args = _try_json(fn.get('arguments', '{}'))
            if not isinstance(args, dict):
                args = {'raw': args}
            parts.append({'functionCall': {'name': fn.get('name', ''), 'args': args}})

        text = msg.get('content', '')
        if isinstance(text, str) and text.strip():
            parts.append({'text': text})

        if not parts:
            parts = [{'text': ''}]

        return {'role': 'model', 'parts': parts}

    if role == 'tool':
        response_obj = _try_json(msg.get('content', ''))
        if not isinstance(response_obj, dict):
            response_obj = {'output': response_obj}

        return {
            'parts': [
                {
                    'functionResponse': {
                        'name': msg.get('name', ''),
                        'response': response_obj,
                    }
                }
            ]
        }

    return {'role': 'user', 'parts': [{'text': msg.get('content', '')}]}

def convert_row(row: Dict[str, Any]) -> Dict[str, Any]:
    messages = row.get('messages', [])
    system_text = ''
    if messages and messages[0].get('role') == 'system':
        system_text = messages[0].get('content', '')
        messages = messages[1:]

    out = {'contents': [_msg_to_content(m) for m in messages]}

    if system_text:
        out['system_instruction'] = {'parts': [{'text': system_text}]}

    tools = _to_tools(row.get('tools', []))
    if tools:
        out['tools'] = tools

    return out

In [ ]:
count = 0
with SRC_FILE.open('r', encoding='utf-8') as fin, OUT_FILE.open('w', encoding='utf-8') as fout:
    for line in fin:
        row = json.loads(line)
        out = convert_row(row)
        fout.write(json.dumps(out, ensure_ascii=False) + '\n')
        count += 1

print('Wrote rows:', count)
print('File size bytes:', OUT_FILE.stat().st_size)

In [ ]:
# Quick sanity check on first converted row
with OUT_FILE.open('r', encoding='utf-8') as f:
    first = json.loads(next(f))

print('Keys:', sorted(first.keys()))
print('Num contents:', len(first.get('contents', [])))
print('Has tools:', 'tools' in first)
print('First content role:', first.get('contents', [{}])[0].get('role'))

## Next Step in Google Cloud

1. Upload output JSONL to GCS.
2. In Model Garden / Agent Platform, launch supervised fine-tuning and select function-calling tuning flow.
3. Use the generated JSONL as training file (and optional validation file).